# FMCW inference
Choose one of the three validation-selected representatives. No training or calibration occurs. The example is a training scene.


In [ ]:
from pathlib import Path
import sys
try:
    from google.colab import drive
except ImportError:
    PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'training/fmcw_training').is_dir())
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/fmcw-mitigation-detection')
sys.path.insert(0, str(PROJECT/'training'))
print(PROJECT)
# Install dependencies once if needed:
# %pip install -r {PROJECT/'training/requirements.txt'}


In [ ]:
MODEL = 'full'  # 'unconditioned' or 'unet_iq'
DEVICE = 'cpu'
THRESHOLD = 0.5  # Main paper comparison; historical calibration is separate.


In [ ]:
import json
import numpy as np
from predictor import Predictor
with np.load(PROJECT/'training/example_training_scene.npz', allow_pickle=False) as example:
    y = example['y'].copy()
    radar = json.loads(str(example['radar_json']))
predictor = Predictor(model=MODEL, device=DEVICE, threshold=THRESHOLD)
result = predictor(y, **radar, diagnostic=True)
print('Model:', predictor.architecture, 'Seed:', predictor.seed, 'Epoch:', predictor.epoch)
print('Threshold:', result['threshold'], 'Detected ranges:', result['detected_m'])


In [ ]:
import matplotlib.pyplot as plt
r = result['distance_m']; valid = result['valid']
fig, axes = plt.subplots(1,2,figsize=(12,3))
axes[0].plot(r[valid],abs(result['complex_reconstruction'][valid]))
axes[0].set_ylabel('Reconstructed FFT amplitude (relative)')
axes[1].plot(r[valid],result['presence'][valid])
axes[1].axhline(result['threshold'],color='grey',linestyle='--')
axes[1].set_ylabel('Presence score')
for ax in axes:
    ax.set_xlabel('Range (m)');ax.grid(alpha=.2)
plt.tight_layout();plt.show()


For complete test evaluation, use `python training/evaluate.py --model all`. The nine-run paper means are separate saved results; this notebook displays one representative at a time.
